In [ ]:
# --------------------------------------------------------------------------
# Imports
# --------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    log_loss
)

# Fix the seed once so the split, the scratch model, and sklearn's solver
# all behave reproducibly across runs.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# --------------------------------------------------------------------------
# Load dataset
# --------------------------------------------------------------------------
# Make sure creditcard.csv (Kaggle "Credit Card Fraud Detection" dataset)
# is in the same folder as this notebook.
df = pd.read_csv("creditcard.csv")

print("Dataset shape:", df.shape)
display(df.head())

print("\nClass distribution (0 = genuine, 1 = fraud):")
print(df["Class"].value_counts())
print("\nClass distribution (%):")
print((df["Class"].value_counts(normalize=True) * 100).round(4))

# The dataset is heavily imbalanced - visualize it so the scale is obvious
# df["Class"].value_counts().plot(kind="bar", title="Class Distribution")
# plt.xlabel("Class (0 = Genuine, 1 = Fraud)")
# plt.ylabel("Number of Transactions")
# plt.show()


In [ ]:
# --------------------------------------------------------------------------
# Feature/target split, scaling, train-test split
# --------------------------------------------------------------------------
X = df.drop("Class", axis=1)
y = df["Class"].values  # plain numpy array, so the scratch model never has to worry about pandas indices

# Standardize features: gradient descent converges far more reliably and
# quickly when every feature is on a comparable scale.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Stratify on y so the (very rare) fraud class keeps the same ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


In [ ]:
# --------------------------------------------------------------------------
# Logistic Regression implemented from scratch
#   - Optimizer : batch gradient descent
#   - Loss      : binary cross-entropy (log loss)
#   - Optional L1 / L2 regularization (reused again in Part 3)
# --------------------------------------------------------------------------
class LogisticRegressionScratch:
    """
    Parameters
    ----------
    learning_rate : step size used for every gradient descent update.
    epochs        : number of full passes over the training data.
    regularization: None, "l1", or "l2". None reproduces plain logistic
                    regression (used in Parts 1 and 2).
    lambda_       : regularization strength; ignored when regularization=None.
    """

    def __init__(self, learning_rate=0.01, epochs=1000,
                 regularization=None, lambda_=0.01):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.regularization = regularization
        self.lambda_ = lambda_
        self.weights = None
        self.bias = 0.0
        self.loss_history = []

    def sigmoid(self, z):
        # Clip z so exp() never overflows on very large/small inputs
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def compute_loss(self, y, predictions):
        # Keep predictions away from exactly 0 or 1 so log() is always finite
        eps = 1e-15
        predictions = np.clip(predictions, eps, 1 - eps)

        # Binary cross-entropy loss
        loss = -np.mean(
            y * np.log(predictions) + (1 - y) * np.log(1 - predictions)
        )

        # Add the regularization penalty to the reported loss. Only the
        # weights are penalized, never the bias term.
        if self.regularization == "l2":
            loss += (self.lambda_ / (2 * len(y))) * np.sum(self.weights ** 2)
        elif self.regularization == "l1":
            loss += (self.lambda_ / len(y)) * np.sum(np.abs(self.weights))

        return loss

    def fit(self, X, y, verbose=False):
        n_samples, n_features = X.shape

        # Zero-initialized weights are fine here: the BCE loss for logistic
        # regression is convex, so there is no symmetry-breaking problem
        # like the one that rules out zero-init for neural networks.
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.loss_history = []

        for epoch in range(self.epochs):
            # --- Forward pass ---
            linear = np.dot(X, self.weights) + self.bias
            predictions = self.sigmoid(linear)

            # --- Gradients of the BCE loss w.r.t. weights and bias ---
            error = predictions - y
            dw = np.dot(X.T, error) / n_samples
            db = np.mean(error)

            # --- Add the regularization gradient term ---
            if self.regularization == "l2":
                dw += (self.lambda_ / n_samples) * self.weights
            elif self.regularization == "l1":
                dw += (self.lambda_ / n_samples) * np.sign(self.weights)

            # --- Gradient descent update ---
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            # --- Track loss for the convergence plot ---
            loss = self.compute_loss(y, predictions)
            self.loss_history.append(loss)

            if verbose and epoch % 100 == 0:
                print(f"Epoch {epoch:4d} | Loss = {loss:.6f}")

        return self

    def predict_proba(self, X):
        linear = np.dot(X, self.weights) + self.bias
        return self.sigmoid(linear)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


In [ ]:
# --------------------------------------------------------------------------
# Shared evaluation helpers (avoid repeating the same metric calls everywhere)
# --------------------------------------------------------------------------
def evaluate_classifier(y_true, y_pred, y_prob):
    """Return a dict of standard binary-classification metrics."""
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob)
    }


def print_results(title, results, y_true, y_pred):
    """Pretty-print a metrics dict plus its confusion matrix."""
    print(f"--- {title} ---")
    for metric, value in results.items():
        print(f"{metric}: {value:.6f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


In [ ]:
# --------------------------------------------------------------------------
# Train the from-scratch model (no regularization) - this is the Part 1 model
# --------------------------------------------------------------------------
model_scratch = LogisticRegressionScratch(learning_rate=0.01, epochs=1000)
model_scratch.fit(X_train, y_train, verbose=True)


In [ ]:
y_pred_scratch = model_scratch.predict(X_test)
y_prob_scratch = model_scratch.predict_proba(X_test)

scratch_results = evaluate_classifier(y_test, y_pred_scratch, y_prob_scratch)
print_results("Scratch Logistic Regression", scratch_results, y_test, y_pred_scratch)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(model_scratch.loss_history)
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Gradient Descent Convergence (Scratch Model)")
plt.grid()
plt.show()


In [ ]:
# --------------------------------------------------------------------------
# Scikit-learn baseline (same train/test split, unregularized)
# --------------------------------------------------------------------------
model_sklearn = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=RANDOM_STATE)
model_sklearn.fit(X_train, y_train)

y_pred_sklearn = model_sklearn.predict(X_test)
y_prob_sklearn = model_sklearn.predict_proba(X_test)[:, 1]

sklearn_results = evaluate_classifier(y_test, y_pred_sklearn, y_prob_sklearn)
print_results("Scikit-learn Logistic Regression", sklearn_results, y_test, y_pred_sklearn)


In [ ]:
comparison = pd.DataFrame({
    "Metric": list(scratch_results.keys()),
    "Scratch": list(scratch_results.values()),
    "Scikit-learn": list(sklearn_results.values())
})
display(comparison)


In [ ]:
# --------------------------------------------------------------------------
# Approximate an iteration-by-iteration loss curve for the sklearn model
# (lbfgs doesn't expose one directly) so it can be compared against the
# scratch model's convergence behavior.
# --------------------------------------------------------------------------
max_iters_to_probe = range(1, 51)
sklearn_loss_history = []

model_sklearn_conv = LogisticRegression(
    solver="lbfgs", warm_start=True, max_iter=1, random_state=RANDOM_STATE
)

for n_iter in max_iters_to_probe:
    model_sklearn_conv.max_iter = n_iter
    model_sklearn_conv.fit(X_train, y_train)
    train_prob = model_sklearn_conv.predict_proba(X_train)[:, 1]
    sklearn_loss_history.append(log_loss(y_train, train_prob))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(model_scratch.loss_history)
axes[0].set_title("Scratch Model (Gradient Descent)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary Cross-Entropy Loss")
axes[0].grid()

axes[1].plot(list(max_iters_to_probe), sklearn_loss_history, color="orange")
axes[1].set_title("Scikit-learn Model (lbfgs)")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Binary Cross-Entropy Loss")
axes[1].grid()

plt.suptitle("Convergence Behavior: Scratch GD vs. Scikit-learn lbfgs")
plt.tight_layout()
plt.show()

print(f"Scikit-learn (lbfgs) reported convergence in {model_sklearn.n_iter_[0]} iterations,")
print(f"versus {model_scratch.epochs} epochs for the scratch gradient descent model.")


In [ ]:
# --------------------------------------------------------------------------
# Train three variants of the SAME scratch class: no penalty, L1, L2
# --------------------------------------------------------------------------
regularized_models = {
    "No Regularization": LogisticRegressionScratch(learning_rate=0.01, epochs=1000),
    "L1": LogisticRegressionScratch(learning_rate=0.01, epochs=1000, regularization="l1", lambda_=0.1),
    "L2": LogisticRegressionScratch(learning_rate=0.01, epochs=1000, regularization="l2", lambda_=0.1),
}

for name, model in regularized_models.items():
    model.fit(X_train, y_train)


In [ ]:
regularization_results = []

for name, model in regularized_models.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)
    metrics = evaluate_classifier(y_test, pred, prob)
    metrics["Model"] = name
    # Count weights pushed (near) exactly to zero - highlights L1's
    # sparsity-inducing effect versus L2's smooth shrinkage.
    metrics["Near-Zero Weights (<1e-3)"] = int(np.sum(np.abs(model.weights) < 1e-3))
    regularization_results.append(metrics)

regularization_df = pd.DataFrame(regularization_results).set_index("Model")
display(regularization_df)


In [ ]:
# --------------------------------------------------------------------------
# model_sklearn (trained in Part 2) is reused as the "Normal" baseline here
# instead of training an identical unweighted model a second time.
# --------------------------------------------------------------------------
model_balanced = LogisticRegression(
    max_iter=1000, solver="lbfgs", class_weight="balanced", random_state=RANDOM_STATE
)
model_balanced.fit(X_train, y_train)

prob_balanced = model_balanced.predict_proba(X_test)[:, 1]
pred_balanced = (prob_balanced >= 0.5).astype(int)

imbalance_results = pd.DataFrame({
    "Normal (Unweighted)": evaluate_classifier(y_test, y_pred_sklearn, y_prob_sklearn),
    "Class Weighted (Balanced)": evaluate_classifier(y_test, pred_balanced, prob_balanced)
})
display(imbalance_results)


In [ ]:
probabilities = prob_balanced  # reuse the probabilities already computed above

thresholds = np.arange(0.01, 1.0, 0.01)  # a finer sweep than 0.1 steps
threshold_results = []

for threshold in thresholds:
    predictions = (probabilities >= threshold).astype(int)
    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.iloc[::10])  # every 10th row, so the table stays readable


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(threshold_df["Threshold"], threshold_df["Precision"], label="Precision")
plt.plot(threshold_df["Threshold"], threshold_df["Recall"], label="Recall")
plt.plot(threshold_df["Threshold"], threshold_df["F1"], label="F1 Score")
plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Effect of Classification Threshold")
plt.legend()
plt.grid()
plt.show()


In [ ]:
best_index = threshold_df["F1"].idxmax()
best_threshold = threshold_df.loc[best_index, "Threshold"]
best_f1 = threshold_df.loc[best_index, "F1"]

print("Best threshold:", round(best_threshold, 2))
print("Best F1 score at that threshold:", round(best_f1, 6))


In [ ]:
optimized_predictions = (probabilities >= best_threshold).astype(int)
optimized_results = evaluate_classifier(y_test, optimized_predictions, probabilities)
print_results(f"Threshold-Optimized Model (threshold = {best_threshold:.2f})",
              optimized_results, y_test, optimized_predictions)
